# 課題と解答例：42_love_dynamics_network

元Notebook: [../42_love_dynamics_network.ipynb](../42_love_dynamics_network.ipynb)

## 6. 課題（実装）

1. `chain_matrix(n, weight)` を作り，一方向の鎖 `node 1 -> node 2 -> ... -> node n` を表す隣接行列を返す．
2. `edge_table(A)` と `graph_from_A(A)` を作り，非対称な `A` で行列の非零要素とnetworkxの矢印が一致することを確認する．
3. `stability_sweep(A, alpha, beta_values)` を作り，`beta` ごとの最大固有値実部をDataFrameにまとめ，安定性が変わる候補を図にする．
4. 研究用Notebookへ進む前に，感情変数，第三者効果，物語イベント，推定法を入れるための列名一覧を作る．実装は基準ネットワークの検証後に1つずつ追加する．

## 解答例

1. 鎖行列の実装例である．

   ```python
   def chain_matrix(n, weight=1.0):
       A = np.zeros((n, n))
       for j in range(n - 1):
           A[j + 1, j] = weight  # node j+1 -> node j+2 in 1-based labels
       return A
   ```

2. 行列から表とグラフを作る．

   ```python
   def edge_table(A):
       return pd.DataFrame([
           {"from": j, "to": i, "weight": A[i, j]}
           for i in range(A.shape[0])
           for j in range(A.shape[1])
           if A[i, j] != 0
       ])

   def graph_from_A(A):
       G = nx.DiGraph()
       G.add_nodes_from(range(A.shape[0]))
       for _, row in edge_table(A).iterrows():
           G.add_edge(int(row["from"]), int(row["to"]), weight=row["weight"])
       return G
   ```

3. 安定性スイープの例である．

   ```python
   def stability_sweep(A, alpha, beta_values):
       rows = []
       for beta in beta_values:
           M = -alpha * np.eye(A.shape[0]) + beta * A
           eig = np.linalg.eigvals(M)
           rows.append({"beta": beta, "max_real": eig.real.max()})
       return pd.DataFrame(rows)
   ```

   `max_real` が0を横切る付近が安定性変化の候補である．

4. 研究用の列名例である．

   ```python
   columns = ["scene", "time", "speaker", "target", "emotion_score", "event_label", "third_person", "note"]
   ```

   対象作品を決めた後，この列を埋められる観測規則を定める．